#### 1.0 Libraries and directories

In [ ]:
import ee 
import geemap
import geopandas as gpd
import pandas as pd
import datetime as dt
import pprint as pp
from shapely.geometry import shape

ee.Authenticate()
ee.Initialize(project='ee-green-by-another-name')

roi_name = 'YKF_sub1'
level = 'sr' # 1) 'sr': surface reflectance 2) 'toa' for top of atmosphere
resample_res = 30 
resample_method = 'bilinear'
n_dates = 10 # Number of observation dates (might be less if there aren't enough footprints)
s2_cloud_threshold = 20 # Cloud probability threshold for Sentinel-2 pixels. 
band_dict = {'Sentinel2_sr': ['B2', #Blue
                             'B3', #Green
                             'B4', #Red
                             'B8'], #NIR
             'LandSat8_sr': ['SR_B2', #Blue
                           'SR_B3', #Green
                           'SR_B4', #Red
                           'SR_B5'], #NIR
            'Sentinel2_toa': ['B2', 
                              'B3',
                              'B4',
                              'B8'],
            'LandSat8_toa': ['B2',
                             'B3',
                             'B4',
                             'B5']
}

image_footprints_path = f'./data/overlap_dates_for_roi/{roi_name}_overlap_dates.shp'
best_image_dates = gpd.read_file(image_footprints_path) 
est_utm = f'EPSG:{best_image_dates.estimate_utm_crs().to_epsg()}' # Have to convert pyproj object into literal string for ee


In [3]:
best_image_dates[['date', 'per_cover']].head(20)

,date,per_cover
0,2019-05-16,81.0
1,2021-07-01,81.0
2,2017-09-08,80.0
3,2021-09-12,66.0
4,2021-08-02,37.0
5,2020-05-18,34.0
6,2018-07-07,29.0


#### 2.0 Select target dates

In [4]:
best_image_dates['date_as_dt'] = pd.to_datetime(best_image_dates['date'])
# Incase we decide to analyze seasonality
early = best_image_dates[best_image_dates['date_as_dt'].dt.month == 6]
early.head(10)
late = best_image_dates[best_image_dates['date_as_dt'].dt.month == 8]
late.head(10)

best_image_dates['date_plus_1d'] = best_image_dates['date_as_dt'] + pd.Timedelta(days=1)
#best_image_dates = best_image_dates[best_image_dates['date_as_dt'] > pd.to_datetime('2019-01-01')]
best_image_dates = best_image_dates[0:n_dates]

#print(best_image_dates[['date', 'per_cover']])

In [6]:
######################################
# Misc. helper functions
######################################
def convert_gpd_geom_to_ee(geom):
        """
        Takes a geopandas geom object and coverts it to an Earth Engine polygon
        """
        coords = list(geom.exterior.coords)
        coords_list = [[x, y] for x, y in coords]
        return ee.Geometry.Polygon(coords_list, proj='EPSG:4326')

def add_1d_to_date(date: str):
        date_plus_1d = pd.to_datetime(date) + pd.Timedelta(days=1)
        date_plus_1d.strftime('%Y-%m-%d')
        return date_plus_1d

#########################################
# Part I: Functions to find the Sentinel-2 and Landsat8 image collections
#########################################

def find_col(footprint: gpd.GeoSeries, level: str, satellite: str):

    date = footprint['date']
    date_plus_1d = add_1d_to_date(date)
    polygon = convert_gpd_geom_to_ee(footprint['geometry'])
    
    
    if level == 'sr' and satellite == 'S2':
        asset_string = 'COPERNICUS/S2_SR_HARMONIZED'
    elif level == 'sr' and satellite == 'LS8':
         asset_string = 'LANDSAT/LC08/C02/T1_L2'
    elif level == 'toa' and satellite == 'LS8':
        asset_string = 'LANDSAT/LC08/C02/T1_TOA'
    elif level == 'toa' and satellite == 'S2':
        asset_string = 'COPERNICUS/S2_HARMONIZED'
    else:
        print(f'ERROR: level arg should be "sr" or "toa" not {level}')
    
    col = (
        ee.ImageCollection(asset_string)
        .filterDate(date, date_plus_1d)
        .filterBounds(polygon)
    )

    if col.size().getInfo() == 0:
        print(f'ERROR: No {satellite} images found for {date} with {asset_string} processing level')
        return None, None
    
    return col, polygon

#########################################
# Part II: Functions to select bands and rescale numerical values
#########################################

def fetch_rescale_imgs(s2_col: ee.ImageCollection,
                       polygon: ee.Geometry,
                       bands: list,
                       satellite: str):
    """
    Generates a single image mosiac with desired bands
    Rescales the bands to match (0-1) surface reflectance range
    TODO: Is rescaling different for TOA??
    """
        
    img = (s2_col.select(bands)
              .mosaic()
              .clip(polygon))
    
    def rescale_s2(img):
        rescaled_bands = img.divide(10_000)
        return rescaled_bands
    
    def rescale_ls8(img):
         rescaled_bands = img.multiply(0.0000275).add(-0.2)
         return rescaled_bands
    
    if satellite == 'S2':
         out_img = rescale_s2(img)
    elif satellite == 'LS8':
        out_img = rescale_ls8(img)
    else:
         print('ERROR: specify satellite as S2 or LS8')
    
    return out_img

#########################################
# Part III: Functions to produce individual cloud masks
#########################################
  
def make_s2_cloud_mask(footprint: gpd.GeoSeries, 
                       s2_col: ee.ImageCollection, 
                       s2_cloud_threshold: int):
    """
    Produces a binary cloud mask from the Copernicus Cloud Probability and Sentinel-2 SCL (Scene Classification Layer)
    """
    date = footprint['date']
    date_plus_1d = add_1d_to_date(date)
    polygon = convert_gpd_geom_to_ee(footprint['geometry'])

    s2_cloud_prob_string = 'COPERNICUS/S2_CLOUD_PROBABILITY'
    s2_clouds = (ee.ImageCollection(s2_cloud_prob_string)
                 .filterBounds(polygon)
                 .filterDate(date, date_plus_1d)
                 .mosaic()
                 .clip(polygon))
    
    # Select the SCL band from the Sentinel-2 image collection
    s2_scl = (s2_col.select('SCL')
              .mosaic()
              .clip(polygon))
    
    clouds_binary = s2_clouds.select('probability').gt(s2_cloud_threshold).rename('cl_binary')
    s2_shaddow_mask = s2_scl.eq(3)
    s2_cirrus_mask = s2_scl.eq(10) 
    s2_full_mask = clouds_binary.Or(s2_shaddow_mask).Or(s2_cirrus_mask)

    return s2_full_mask

def make_ls8_cloud_mask(ls8_col: ee.ImageCollection, polygon: ee.Geometry):
    """
    Generates a cloud mask for Landsat 8 images using QA_PIXEL bit flags.
    """
    ls8_qa = (ls8_col
              .select('QA_PIXEL')
              .mosaic()
              .clip(polygon))
    
    # Define bitmasks for the conditions
    cloud_bit_mask = 1 << 3        # Bit 3: Cloud
    cloud_shadow_bit_mask = 1 << 4  # Bit 4: Cloud Shadow
    snow_bit_mask = 1 << 5         # Bit 5: Snow
    cirrus_bit_mask = 1 << 2       # Bit 2: Cirrus
    dilated_cloud_bit_mask = 1 << 1 # Bit 1: Dilated Cloud

    # Combine all bitmasks into one
    bitmask = (cloud_bit_mask
               | cloud_shadow_bit_mask
               | snow_bit_mask
               | cirrus_bit_mask
               | dilated_cloud_bit_mask)

    # Create the mask where any of the bits are set
    ls8_full_mask = ls8_qa.bitwiseAnd(bitmask).neq(0)

    return ls8_full_mask

def reduce_mask_resolution(mask: ee.Image, resample_res: int, est_utm: str):
    """Reduces the resolution of cloud masks"""

    original_crs = mask.projection()
    mask_reproj = mask.reproject(
        crs=ee.Projection(est_utm),
        scale=resample_res
    )
    mask_repoj_reduced = mask_reproj.reduceResolution(
        reducer=ee.Reducer.mean()
    )

    return mask_repoj_reduced

#########################################
# Part IV: Functions to generate common masks and apply it to images
#########################################

def combine_sieve_dilate_masks(s2_mask, ls8_mask, est_utm, resample_res):
     
     size_threshold = 50 # Maximum number of pixels to ignore (sieve out) of the cloud max
     dilation_radius = 500 # The distance (meters) to dilate the clouds for a more conservative cloud mask. 
     
     # Check that both masks are in local UTM
     if str(ls8_mask.projection().getInfo()) is not est_utm:
        reproj_mask = ls8_mask.reproject(
            crs = ee.Projection(est_utm),
            scale = resample_res
        )

     combined_mask = s2_mask.Or(ls8_mask)

     connected_pixels = reproj_mask.connectedPixelCount(maxSize=100, eightConnected=True)
     sieved_mask = reproj_mask.updateMask(connected_pixels.gte(size_threshold))
     dilation_kernel = ee.Kernel.circle(radius=dilation_radius, units='meters', normalize=False)
     dilated_mask = sieved_mask.focal_max(kernel=dilation_kernel, iterations=1)

     return dilated_mask

def resample_img_then_mask(img: ee.Image, 
                           common_mask: ee.Image, 
                           est_utm: str, 
                           resample_res: int,
                           resample_method: str):
    """
    Resample to match the common cloud mask, then mask the image
    """

    reproj = img.reproject(
         crs=ee.Projection(est_utm),
         scale=resample_res
    )
    resamp = reproj.resample(resample_method)
    masked = resamp.updateMask(common_mask.neq(1))
    
    return masked

#########################################
# Part V: Main processsing & export functions
#########################################

def s2_processor(footprint: gpd.GeoSeries, 
                level: str, 
                bands: list, 
                resample_res: int,
                est_utm: str, 
                s2_cloud_threshold: int):
    """
    Main function to process and export the Sentinel-2 image mosaic and cloud mask
    """
    s2_col, footprint_geom_ee = find_col(footprint, level, satellite='S2')
    if s2_col is None:
        return None, None
    else: 
        s2_img = fetch_rescale_imgs(s2_col, footprint_geom_ee, bands, satellite='S2')
        s2_cloud_mask = make_s2_cloud_mask(footprint, s2_col, s2_cloud_threshold)
        s2_cloud_mask = reduce_mask_resolution(s2_cloud_mask, resample_res, est_utm)

        return s2_img, s2_cloud_mask
    
def ls8_processor(footprint: gpd.GeoSeries, 
                  level: str, 
                  bands: list,
                  resample_res: int,
                  est_utm):
    """
    Main function to process and export LandSat-8 images
    """
    ls8_col, footprint_geom_ee = find_col(footprint, level, satellite='LS8')
    if ls8_col is None:
         return None, None
    else:
        ls8_img = fetch_rescale_imgs(ls8_col, footprint_geom_ee, bands, satellite='LS8')
        ls8_cloud_mask = make_ls8_cloud_mask(ls8_col, footprint_geom_ee)
        # Don't bother resampling Landsat, if resample resolution = 30 meters
        if resample_res != 30:
            ls8_cloud_mask = reduce_mask_resolution(ls8_cloud_mask, resample_res, est_utm)
        else:
            return ls8_img, ls8_cloud_mask
    

def image_exporter(masked_s2: ee.Image, 
                   masked_ls8: ee.Image, 
                   polygon: ee.Geometry, 
                   footprint: gpd.GeoSeries,
                   resample_res: int,
                   resample_method: str):

    s2_export = ee.batch.Export.image.toDrive(
         image=masked_s2,
         description=f'Sentinel2-{footprint['date']}_{roi_name}_resampled_{resample_method}{resample_res}',
         fileNamePrefix=f'Sentinel2-{footprint['date']}_{roi_name}_resampled_{resample_method}{resample_res}',
         folder='scrap1',
         scale=30,
         region=polygon,
         crs='EPSG:4326',
         fileFormat='GeoTIFF',
         maxPixels=1e13
    )

    s2_export.start()
    print('Exporting Sentinel-2')

    ls8_export = ee.batch.Export.image.toDrive(
         image=masked_ls8,
         description=f'Landsat8-{footprint['date']}_{roi_name}_resampled_{resample_method}{resample_res}',
         fileNamePrefix=f'Landsat8-{footprint['date']}_{roi_name}_resampled_{resample_method}{resample_res}',
         folder='scrap1',
         scale=30,
         region=polygon,
         crs='EPSG:4326',
         fileFormat='GeoTIFF',
         maxPixels=1e13
    )

    ls8_export.start()
    print('Exporting Landsat8')

def main_processor(footprint, level, band_dict, resample_res, resample_method, est_utm, s2_cloud_threshold):

    polygon = convert_gpd_geom_to_ee(footprint['geometry'])

    if level == 'toa':
        ls8_bands_key = 'LandSat8_toa'
        s2_bands_key = 'Sentinel2_toa'
    elif level == 'sr':
        ls8_bands_key = 'LandSat8_sr'
        s2_bands_key = 'Sentinel2_sr'
    else:
        print('Error specify proper level')
    
    s2_img, s2_cloud_mask = s2_processor(footprint=footprint,
                                        level=level,
                                        bands=band_dict[s2_bands_key],
                                        resample_res=resample_res,
                                        est_utm=est_utm,
                                        s2_cloud_threshold=s2_cloud_threshold)
    
    ls8_img, ls8_cloud_mask = ls8_processor(footprint=footprint,
                                            level=level,
                                            bands=band_dict[ls8_bands_key],
                                            resample_res=resample_res,
                                            est_utm=est_utm)
    
    if s2_img is None or ls8_img is None:
         return None
    else:
        common_mask = combine_sieve_dilate_masks(s2_cloud_mask, ls8_cloud_mask, est_utm, resample_res)
        masked_s2 = resample_img_then_mask(s2_img, common_mask, est_utm, resample_res, resample_method)
        masked_ls8 = resample_img_then_mask(ls8_img, common_mask, est_utm, resample_res, resample_method)
        pp.pp(masked_s2.projection().getInfo())
        pp.pp(masked_ls8.projection().getInfo())

        image_exporter(masked_s2, masked_ls8, polygon, footprint, resample_res, resample_method)


In [7]:
for idx, row in best_image_dates.iterrows():
    print(row['date'])
    main_processor(
        footprint=row,
        level='toa',
        band_dict=band_dict,
        resample_res=resample_res,
        resample_method=resample_method,
        est_utm=est_utm,
        s2_cloud_threshold=s2_cloud_threshold
    )

2019-05-16
{'type': 'Projection', 'crs': 'EPSG:32606', 'transform': [30, 0, 0, 0, -30, 0]}
{'type': 'Projection', 'crs': 'EPSG:32606', 'transform': [30, 0, 0, 0, -30, 0]}
Exporting Sentinel-2
Exporting Landsat8
2021-07-01
{'type': 'Projection', 'crs': 'EPSG:32606', 'transform': [30, 0, 0, 0, -30, 0]}
{'type': 'Projection', 'crs': 'EPSG:32606', 'transform': [30, 0, 0, 0, -30, 0]}
Exporting Sentinel-2
Exporting Landsat8
2017-09-08
{'type': 'Projection', 'crs': 'EPSG:32606', 'transform': [30, 0, 0, 0, -30, 0]}
{'type': 'Projection', 'crs': 'EPSG:32606', 'transform': [30, 0, 0, 0, -30, 0]}
Exporting Sentinel-2
Exporting Landsat8
2021-09-12
{'type': 'Projection', 'crs': 'EPSG:32606', 'transform': [30, 0, 0, 0, -30, 0]}
{'type': 'Projection', 'crs': 'EPSG:32606', 'transform': [30, 0, 0, 0, -30, 0]}
Exporting Sentinel-2
Exporting Landsat8
2021-08-02
{'type': 'Projection', 'crs': 'EPSG:32606', 'transform': [30, 0, 0, 0, -30, 0]}
{'type': 'Projection', 'crs': 'EPSG:32606', 'transform': [30, 0, 